What is  Proteus?
=================

A Python toolkit for solving equations

Find $u:\Omega \rightarrow \mathbb{R}$ satisfying
$$
\nabla \cdot (-a\nabla u) = r(x)
$$
on 
$$
\Omega = [0,1] \times [0,1]
$$
subject to
$$
u|_{\partial \Omega} = 0
$$

First, import some tools. 

Remember to click on the cell and hit shift-enter to execute code.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from proteus import runtime
comm = runtime.start()  # MPI and PETSc, and proteus.log in the working directory
from proteus import (AnalyticalSolutions, Domain, FemTools, Norms, NumericalFlux, NumericalSolution,
                     Quadrature, TransportCoefficients, default_n, default_s, default_so)
import proteus.default_p as physics
import proteus.default_n as numerics
from proteus.TransportCoefficients import PoissonEquationCoefficients

Define the equation

In [ ]:
def r(x):
    return (-32.*np.pi*2*
            np.sin(np.pi*2*x[...,0])*
            np.sin(np.pi*2*x[...,1])*
            np.cos(np.pi*2*x[...,0])*
            np.cos(np.pi*2*x[...,1]))
def a(x):
    return np.array([[1.0,0.0],
                     [0.0,1.0]])
physics.coefficients = PoissonEquationCoefficients([a],[r]) 
physics.name="PoissonDots"

Define the domain and boundary conditions

In [ ]:
physics.nd = 2
physics.domain =  Domain.RectangularDomain(L=[1.0,1.0],
                                           x=[0.0,0.0])
def getDBC(x,flag):
    if flag in [physics.domain.boundaryTags['left'],
                physics.domain.boundaryTags['top'],
                physics.domain.boundaryTags['right'],
                physics.domain.boundaryTags['bottom']]:
        return lambda x,t: 0.0
        
physics.dirichletConditions = {0:getDBC}

Configure the numerics

In [ ]:
numerics=default_n
numerics.femSpaces = {0:FemTools.P1}
numerics.elementQuadrature = Quadrature.SimplexGaussQuadrature(physics.nd,3)
numerics.elementBoundaryQuadrature = Quadrature.SimplexGaussQuadrature(physics.nd-1,3)
numerics.nn = 41
numerics.nLevels = 2
numerics.numericalFluxType = NumericalFlux.Diffusion_SIPG_exterior
so = default_so
so.name = physics.name
so.sList=[default_s]
so.tnList = [0,1]

Solve the problem

In [ ]:
ns = NumericalSolution.NS_base(so, [physics], [numerics], so.sList, runtime.RunOptions())
failed = ns.calculateSolution('run1')
assert(not failed)

Plot the solution

In [ ]:
model = ns.modelList[0].levelModelList[-1]
plt.figure()
plt.tricontourf(model.mesh.nodeArray[:,0],
                model.mesh.nodeArray[:,1],
                model.mesh.elementNodesArray,
                model.u[0].dof)
plt.colorbar()
plt.gca().set_aspect('equal')

Compare with the exact solution

With $a = I$ the source is
$$
r(x) = -64\pi \sin(2\pi x)\sin(2\pi y)\cos(2\pi x)\cos(2\pi y) = -16\pi \sin(4\pi x)\sin(4\pi y)
$$
and, since $\Delta\left[\sin(4\pi x)\sin(4\pi y)\right] = -32\pi^2 \sin(4\pi x)\sin(4\pi y)$ and the product vanishes on $\partial\Omega$,
$$
u(x) = -\frac{1}{2\pi}\sin(4\pi x)\sin(4\pi y)
$$

In [ ]:
import math
class PoissonSolution(AnalyticalSolutions.SteadyState):
    def uOfX(self, x):
        return -np.sin(4*np.pi*x[0])*np.sin(4*np.pi*x[1])/(2*np.pi)

def L2error(model, exact):
    """||u_h - u|| in L2(Omega), with the model's element quadrature"""
    return Norms.L2errorSFEMvsAF(exact, model.q['x'], model.q[('dV_u', 0)], model.q[('u', 0)], T=1.0)

error=0
error_last=0
for model in ns.modelList[0].levelModelList:
    error_last=error
    error = L2error(model, PoissonSolution())
    print("P1, nDOF = %d: L2 error %.3e" % (model.u[0].dof.shape[0], error))
p = math.log(error_last/error)/math.log(2.0)
print("p = %f" % (p,))
assert p >= 1.95

A Python toolkit for research on ***continuum models***. The next cell defines a new continuum model (a nonlinear Poisson equation)
$$
\nabla \cdot (-a\nabla \phi(u)) + r(x) = 0, \qquad \phi(u) = u^2 + 10u
$$
with the same $a$, $r$ and boundary condition. This class adds $r$ to the residual, where
`PoissonEquationCoefficients` moved it to the right-hand side.

In [ ]:
class NonlinearPoisson(TransportCoefficients.TC_base):
    def __init__(self,A,rofx):
        TransportCoefficients.TC_base.__init__(self, 
                         nc=1, 
                         variableNames=['u'],
                         diffusion = {0:{0:{0:'constant'}}},
                         potential = {0:{0:'nonlinear'}},
                         reaction  = {0:{0:'constant'}})
        self.A=A;
        self.rofx=rofx
    def evaluate(self,t,c):
        c[('a',0,0)][...,0] = self.A[0][0]
        c[('a',0,0)][...,3] = self.A[1][1]
        c[('r',0)][:]       = self.rofx(c['x'][:])
        c[('phi',0)][:]     = c[('u',0)]**2+10.0*c[('u',0)]
        c[('dphi',0,0)][:]  = 2.0*c[('u',0)] + 10.0

In [ ]:
physics.coefficients = NonlinearPoisson([[1.0,0.0],[0.0,1.0]], r)
numerics.tolFac=0.0
numerics.nl_atol_res = 1.0e-12
numerics.numericalFluxType = None

In [ ]:
ns = NumericalSolution.NS_base(so, [physics], [numerics], so.sList, runtime.RunOptions())
failed = ns.calculateSolution('run2')
assert(not failed)

In [ ]:
model = ns.modelList[0].levelModelList[-1]
plt.figure()
plt.tricontourf(model.mesh.nodeArray[:,0],
                model.mesh.nodeArray[:,1],
                model.mesh.elementNodesArray,
                model.u[0].dof)
plt.colorbar()
plt.gca().set_aspect('equal')

Compare with the exact solution

The equation is linear in the potential (it is already in Kirchhoff-transformed form): $\phi$ solves
$$
\nabla \cdot (-\nabla\phi) + r(x) = 0, \qquad \phi|_{\partial\Omega} = \phi(0) = 0,
$$
so $\phi(x) = \frac{1}{2\pi}\sin(4\pi x)\sin(4\pi y)$ (the opposite sign to the first example). Inverting $u^2 + 10u = \phi$ with the root that vanishes on $\partial\Omega$,
$$
u(x) = -5 + \sqrt{25 + \phi(x)}
$$

In [ ]:
class NonlinearPoissonSolution(AnalyticalSolutions.SteadyState):
    def uOfX(self, x):
        phi = np.sin(4*np.pi*x[0])*np.sin(4*np.pi*x[1])/(2*np.pi)
        return -5.0 + np.sqrt(25.0 + phi)

#print("P1, nn = %d: L2 error %.3e" % (numerics.nn, L2error(model, NonlinearPoissonSolution())))
error=0
error_last=0
for model in ns.modelList[0].levelModelList:
    error_last=error
    error = L2error(model, NonlinearPoissonSolution())
    print("P1, nDOF = %d: L2 error %.3e" % (model.u[0].dof.shape[0], error))
p = math.log(error_last/error)/math.log(2.0)
print("p = %f" % (p,))
assert p >= 1.9

A Python toolkit for ***reproducible*** research on continuum models. The nonlinear model above is Python code (a
`TC_base` subclass and `getDBC`): it runs, but it is not data, and nothing outside Proteus can read it. The next cells
write the whole problem (the strong form, its boundary and initial conditions, the exact solution and the
discretization) as a [ymf](https://github.com/proteustoolkit/ymf) spec, a plain-text document with nothing
Proteus-specific in it. Proteus then derives the weak form from it, solves it and archives the result.

The equations are written in ymf's notation: `∇·(−a∇φ) + r = 0` with `φ = u² + 10u` is the equation above, and the
`TC_base`'s potential, diffusion and reaction are read off it.

In [ ]:
import tempfile
from pathlib import Path

from proteus.cli import main as pro    # the `pro` command: pro spec, pro weak, pro run

work = Path(tempfile.mkdtemp(prefix="introduction_ymf_"))   # the spec, its archive and the solutions go here
spec = work / "nonlinear_poisson.ymf"
spec.write_text("""\
ymf_state: discretized
model:
  name: "Nonlinear Poisson equation in Kirchhoff form, unit square"
  physical_model: {provenance: human_specified, processes: [diffusion], assumptions: [steady]}
  strong_form:
    provenance: human_specified
    unknowns: [u]
    equation_formulation: "steady diffusion, nonlinear in the potential"
    equations:
      - "∇·(−a∇φ) + r = 0  in Ω"
    domain: "Ω = [0, 1] × [0, 1]"
    boundary_regions:
      - {name: boundary, where: "∂Ω"}
    boundary_conditions:
      - {region: boundary, variable: u, type: dirichlet, value: 0.0}
    initial_conditions:            # a steady problem: Newton's starting guess
      - {field: u, type: constant, value: 0.0}
    coefficients:
      a: 1.0
      φ: {formula: "u² + 10u"}     # a coefficient may name the unknown
      r: {formula: "−16π sin(4πx) sin(4πy)"}
solution_paths:
  analytical:
    - name: kirchhoff
      provenance: human_specified
      method: closed_form
      solution:
        formula: "u = −5 + sqrt(25 + sin(4πx) sin(4πy)/(2π))"
  discretizations:
    - name: P1
      provenance: human_specified
      finite_element: {fields: {family: CG, order: 1}}
      mesh: {cells: [20, 40, 80]}
      solver: {type: nonlinear, nonlinear_solver: newton, linear_solver: superlu, tolerance: 1.0e-12}
""", encoding="utf-8")
print(spec.read_text(encoding="utf-8"))

`pro spec` checks the document and says how far it has got: `well-posed` (the problem is stated), `discretized`
(every discretization gives a mesh) or `closed` (every output it plans is computed and archived). ymf reads the
document; Proteus only says whether it can solve it.

In [ ]:
pro(["spec", str(spec)]);

`pro weak` derives the weak form Proteus will solve: the residual sorted into Proteus's slots,
$\partial m/\partial t + \nabla\cdot(f - a\nabla\phi) + r + H = 0$, multiplied by a test function $w$ and integrated by
parts. ymf recognizes $\nabla\cdot(-a\nabla\phi(u))$ as diffusion of the nonlinear potential $\phi(u) = u^2 + 10u$
(the Kirchhoff form), as the `TC_base` above declares with `potential = {0: {0: 'nonlinear'}}`.

In [ ]:
import yaml
pro(["weak", str(spec)]);
weak = yaml.safe_load((work / "nonlinear_poisson.weak.ymf").read_text(encoding="utf-8"))["model"]["weak_forms"][0]
print("\n" + weak["bilinear"] + "   =   " + weak["linear"].split(": ", 1)[1])

What reaches the solver is plain data: numpy code strings and dependence flags, the same slots as `TC_base`, with
no sympy and nothing of the spec's notation. That is what keeps the spec independent of the solver.

In [ ]:
from pprint import pprint
from ymf import closure
from ymf.symbolic import adr_problem

problem = adr_problem(closure.plain(closure.load(str(spec))[0]))
pprint(problem["adr"]["equations"][0], width=100)

`pro run` solves the discretization on every mesh the spec lists and writes an *archive*: the spec as realized, then
one output per mesh. Each output is keyed by a hash of exactly the input that produced it and records the
transformations (strong form → ADR form → Proteus's spaces, quadrature and solvers), the software versions, its L2
error against the exact solution and the solution itself. On the 1.x solve's meshes (40 and 80 cells: `nn = 41` and
one refinement) the errors agree with it to about 3%: the 1.x cells integrate with quadrature of degree 3, both in the
solve and in the error, and `ADRProblem` with degree 4.

In [ ]:
pro(["run", str(spec), "--outdir", str(work)]);
archive = work / "nonlinear_poisson.archive.ymf"

Rerunning with `--check` recomputes every output and compares it with the archive: the ADR form, the errors and every
array, bitwise.

In [ ]:
pro(["run", str(archive), "--check"]);

### How close is this to a reproducible, Proteus-agnostic model?

The spec holds the whole problem: domain, equations, coefficients (including one that names the unknown), boundary
regions and conditions, the initial condition (Newton's starting guess, since the problem is steady), the exact
solution, the finite element family and order, the meshes and the solver. Nothing in it names a Proteus class: the
weak form, the ADR slots and `C0_AffineLinearOnSimplexWithNodalBasis` are derived and recorded in the archive, not
stated. The archive reproduces bitwise.

What is not there yet:

- **The 1.x objects do not export themselves.** A `TC_base` subclass and `getDBC` are code. The domain does translate
  (next cell); the coefficients and conditions have to be restated in the notation, as above.
- **Weak Dirichlet conditions.** The first example imposes $u = 0$ weakly (`Diffusion_SIPG_exterior`). ymf can say
  so (`imposed: weak`, `penalty`, `form: symmetric`), but its ADR problem refuses a weakly imposed condition in one
  system and `ADRProblem` imposes Dirichlet values strongly, so only the strong-Dirichlet version serializes today.
  That is why the nonlinear example above uses `numericalFluxType = None`.
- **Quadrature** is not stated in the spec: `ADRProblem` uses degree $2k + 2$ (4 for P1), and the 1.x cells use 3.

In [ ]:
from proteus import DomainYMF
print(yaml.safe_dump(DomainYMF.to_ymf(physics.domain), sort_keys=False))

## The same problem from Python operators

The spec can also be posed in Python, with operators that read like the strong form. `proteus.StrongForm` builds
the same document: the objects print themselves in ymf's notation, and solving goes through the spec, so the YAML
problem and the Python problem are solved by the same code.

In [ ]:
from proteus.StrongForm import Box, Coefficient, Problem, Unknown, div, grad, pi, sin, sqrt, x, y

u = Unknown("u")
A = Coefficient("a", 1.0)      # the Python names need not be the spec's:
Φ = Coefficient("φ", u**2 + 10*u)   # a and r are still the 1.x functions,
R = Coefficient("r", -16*pi*sin(4*pi*x)*sin(4*pi*y))   # which the P2 cell below uses

kirchhoff = Problem("Nonlinear Poisson equation in Kirchhoff form, unit square", Box((0, 1), (0, 1)), unknowns=[u],
                    formulation="steady diffusion, nonlinear in the potential")
kirchhoff.equation(div(-A*grad(Φ)) + R == 0)
kirchhoff.dirichlet(u, 0.0, on=kirchhoff.boundary)
kirchhoff.initial(u, 0.0)
kirchhoff.exact(u, -5 + sqrt(25 + sin(4*pi*x)*sin(4*pi*y)/(2*pi)), name="kirchhoff")
kirchhoff.discretization("P1", order=1, cells=[20, 40, 80], tolerance=1e-12)
print(kirchhoff.yaml())

ymf reads both documents to the same plain data, and `pro run` gives the same errors, bitwise. The archive keys
differ, because the key hashes the input as written (`4πx` and `4π x` are different text).

In [ ]:
print("the same plain data as the YAML spec:", kirchhoff.adr() == problem)
python_archive = kirchhoff.run(str(work / "python"), stem="nonlinear_poisson")


def errors(archive):
    _, outputs, _ = closure.load(str(archive), check_state=False)
    return {o["realization"]["cells"]: o["verification"]["l2_errors"]["u"] for o in outputs.values()}


print("the same errors, bitwise:", errors(python_archive) == errors(archive))

A Python toolkit for research on ***numerical methods***. The next cell switches the discretization to a new finite element (P2, or continuous quadratics on triangles with the Lagrange basis).

In [ ]:
physics.coefficients = PoissonEquationCoefficients([a],[r]) 
numerics.femSpaces={0:FemTools.P2}
ns = NumericalSolution.NS_base(so, [physics], [numerics], so.sList, runtime.RunOptions())
failed = ns.calculateSolution('run3')
assert(not failed)

In [ ]:
model = ns.modelList[0].levelModelList[-1]
import matplotlib.tri as tri
triang = tri.Triangulation(model.u[0].femSpace.dofMap.lagrangeNodesArray[:,0],
                model.u[0].femSpace.dofMap.lagrangeNodesArray[:,1])

plt.figure()
plt.tricontourf(triang, model.u[0].dof)
plt.colorbar()
plt.gca().set_aspect('equal')

In [ ]:
#print("P2, nn = %d: L2 error %.3e" % (numerics.nn, L2error(model, PoissonSolution())))
error=0
error_last=0
for model in ns.modelList[0].levelModelList:
    error_last=error
    error = L2error(model, PoissonSolution())
    print("P1, nDOF = %d: L2 error %.3e" % (model.u[0].dof.shape[0], error))
p = math.log(error_last/error)/math.log(2.0)
print("p = %f" % (p,))
assert p >= 2.9

These examples were intended to be simple and general illustrations of some of the main goals of the Proteus project:

- Enabling R&D on new continuum models (PDEs)
- Enabling R&D on new numerical methods for existing models

This notebook directory contains additional tutorials and examples.

https://github.com/proteustoolkit/proteus